# Prompt 7: Garbage Collection in Spark
## Databricks Certified Associate Developer for Apache Spark
### Topic 1 — Apache Spark Architecture & Components (20%)

---

Garbage Collection (GC) is a **top cause of Spark job slowdowns** in production.
Because Spark Executors are JVM processes, they inherit the JVM's GC behaviour.
Understanding GC helps you tune memory, choose the right data structures, and diagnose stalls.

**Exam facts to memorise:**
- Spark Executors run inside a **JVM** — all GC concepts apply (heap, young gen, old gen, GC pauses)
- `spark.executor.memory` = total heap size allocated to each Executor JVM
- **G1GC** (Garbage-First GC) is the recommended collector for Spark since JDK 9+
- High GC time in the Spark UI → Executor GC column shows time per task spent in GC
- Off-heap memory (`spark.memory.offHeap`) bypasses JVM GC entirely
- Caching with serialised storage levels (`MEMORY_ONLY_SER`, `MEMORY_AND_DISK_SER`) reduces GC pressure
- Too many short-lived Java objects = GC pressure = **GC pauses** = slow tasks

## 1. Why GC Matters in Spark

### JVM Memory Layout (inside each Executor)

```
+--------------------------------------------------+
|               Executor JVM Heap                  |
|                                                  |
|  +-----------+  +----------------------------+  |
|  | Young Gen |  |         Old Gen             |  |
|  | (Eden +   |  | (long-lived objects: cached |  |
|  |  Survivor)|  |  RDDs, broadcast vars, etc) |  |
|  +-----------+  +----------------------------+  |
|                                                  |
|  Spark Unified Memory (execution + storage)      |
|  spark.memory.fraction (default 0.6)             |
|  +-----------------+  +----------------------+  |
|  | Execution Memory|  | Storage Memory        |  |
|  | (shuffles,joins)|  | (cached RDDs/DFs)     |  |
|  +-----------------+  +----------------------+  |
|                                                  |
|  User Memory (Python objects, UDF data, etc.)    |
|  spark.memory.fraction complement (default 0.4)  |
+--------------------------------------------------+
```

### Spark Memory Regions

| Region | Fraction | Purpose | GC Impact |
|--------|----------|---------|----------|
| **Unified Memory** (execution + storage) | `spark.memory.fraction` (0.6) | Shuffle buffers, join hash tables, cached DataFrames | High — large objects promoted to Old Gen |
| **User Memory** | 1 - `spark.memory.fraction` (0.4) | Non-Spark JVM objects (UDF closures, Python data, temp vars) | Medium — depends on UDF complexity |
| **Reserved Memory** | Fixed 300 MB | Internal Spark objects | Low |

### The GC Problem in Spark

1. **Short-lived objects** — every row processed creates intermediate Java/Scala objects → fills Young Gen rapidly → frequent **Minor GC** (fast but adds up across millions of rows)
2. **Long-lived cached data** — cached RDD blocks occupy Old Gen → infrequent **Major/Full GC** (slow — may pause ALL threads for seconds)
3. **GC pauses block tasks** — during a Full GC, all Executor threads stop → tasks appear stalled → Spark heartbeat to Driver may time out → false executor failure
4. **Cascading effect** — if GC overhead exceeds 98% of time with <2% heap freed, JVM throws `OutOfMemoryError: GC overhead limit exceeded`

## 2. GC Collectors — Choosing the Right One

| Collector | JVM Flag | Best For | GC Pauses | Throughput |
|-----------|----------|---------|-----------|------------|
| **G1GC** (Garbage-First) | `-XX:+UseG1GC` | **Recommended for Spark** — balances pause time and throughput | Predictable, bounded | High |
| **Parallel GC** (Throughput) | `-XX:+UseParallelGC` | Batch jobs tolerant of occasional long pauses | Longer, unpredictable | Highest |
| **CMS** (Concurrent Mark Sweep) | `-XX:+UseConcMarkSweepGC` | Legacy — deprecated in JDK 9, removed JDK 14 | Short concurrent pauses but degrades with fragmentation | Medium |
| **ZGC** | `-XX:+UseZGC` | Extremely low pause (<1ms) — JDK 15+ | Sub-millisecond | Good |
| **Shenandoah** | `-XX:+UseShenandoahGC` | Low pause; OpenJDK only | Sub-millisecond | Good |

> **Exam note:** The exam tests awareness that GC affects Spark performance and that G1GC is the recommended tuning choice. Deep JVM internals are not tested, but knowing the key flags and the `spark.executor.extraJavaOptions` pattern is important.

### Setting GC options in Spark

```python
# In SparkSession.builder
spark = SparkSession.builder \
    .config('spark.executor.extraJavaOptions',
            '-XX:+UseG1GC '
            '-XX:+PrintGCDetails '
            '-XX:+PrintGCDateStamps '
            '-XX:OnOutOfMemoryError=kill %p') \
    .config('spark.driver.extraJavaOptions',
            '-XX:+UseG1GC') \
    .getOrCreate()

# Via spark-submit
# spark-submit \
#   --conf 'spark.executor.extraJavaOptions=-XX:+UseG1GC -XX:G1HeapRegionSize=16m' \
#   --conf 'spark.driver.extraJavaOptions=-XX:+UseG1GC' \
#   my_app.py
```

## 3. Key GC Tuning Parameters

### Spark Memory Configuration

| Parameter | Default | Effect on GC |
|-----------|---------|-------------|
| `spark.executor.memory` | 1g | Total Executor heap; increase to give GC more room |
| `spark.memory.fraction` | 0.6 | Fraction of heap for execution+storage; lower → more user memory, less GC pressure from cached data |
| `spark.memory.storageFraction` | 0.5 | Within unified memory, fraction protected from eviction for storage; lower → more execution memory |
| `spark.executor.memoryOverhead` | 10% of executor memory (min 384MB) | Off-heap overhead for Python processes, network buffers; increase for PySpark UDFs |
| `spark.memory.offHeap.enabled` | false | Enable Tungsten off-heap; **bypasses JVM GC entirely** |
| `spark.memory.offHeap.size` | 0 | Size of off-heap pool in bytes; requires `offHeap.enabled=true` |

### JVM Flags for G1GC Tuning

| Flag | Recommended | Effect |
|------|-------------|--------|
| `-XX:+UseG1GC` | ✅ | Use G1 collector |
| `-XX:G1HeapRegionSize=16m` | For large heaps (>8GB) | Larger regions = fewer GC events for big objects |
| `-XX:InitiatingHeapOccupancyPercent=35` | Lower than default 45 | Start concurrent GC earlier; prevent Full GC |
| `-XX:ConcGCThreads=4` | = num_cores / 4 | Parallel GC threads |
| `-XX:+PrintGCDetails` | Dev/debug only | Log GC events (parse with GCViewer or GCEasy) |
| `-XX:+HeapDumpOnOutOfMemoryError` | Production | Dump heap for analysis on OOM |
| `-XX:MaxGCPauseMillis=200` | 200ms (G1 target) | Target max GC pause time (G1 honours as best-effort) |

## 4. Identifying GC Overhead in Your Applications

### Spark UI — Where to Look

```
Spark UI → Stages tab → Click a Stage → Task Metrics table

Columns to inspect:
+----------+----------+-----------+-----------+------------+----------+
| Task ID  | Duration | GC Time   | Peak Exec | Shuffle    | Result   |
|          |          |           | Memory    | Read       | Size     |
+----------+----------+-----------+-----------+------------+----------+
| Task 0   | 10s      | 0.5s (5%) | 512MB     | 100MB      | 1KB      |
| Task 1   | 30s      | 25s (83%) | 800MB     | 100MB      | 1KB      |  ← GC problem!
| Task 2   | 11s      | 0.8s (7%) | 490MB     | 100MB      | 1KB      |
+----------+----------+-----------+-----------+------------+----------+

Red flags:
  GC Time > 10% of Task Duration  →  GC pressure
  GC Time > 25% of Task Duration  →  Serious GC problem
  GC Time > 50% of Task Duration  →  Critical — application effectively stalled
```

### Spark UI — Environment tab
- Check `spark.executor.memory` and `spark.executor.extraJavaOptions` are set as expected

### Spark UI — Executor tab
- Peak JVM Memory column shows max heap usage per executor
- If Peak JVM Memory is close to `spark.executor.memory` → heap is too small

### Executor Logs
With `-XX:+PrintGCDetails` in `spark.executor.extraJavaOptions`:
```
[GC (Allocation Failure) [PSYoungGen: 128M->15M(150M)] 256M->145M(512M), 0.0230 secs]
[Full GC (Ergonomics) [PSYoungGen: 15M->0M(150M)] [ParOldGen: 130M->143M(362M)] ...
    [Metaspace: 35M->35M(1085M)], 2.345 secs]   ← Full GC = 2.3 second pause!
```

### GC Log Analysis Tools
- **GCViewer** (open source): visualises GC log files
- **GCEasy** (web): upload GC log, get instant analysis report
- **Spark History Server**: post-job analysis with full task metrics
- **Ganglia / Prometheus + Grafana**: real-time JVM metrics for production clusters

In [ ]:
# Cell 1: Setup — SparkSession with GC-relevant configuration
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, rand
from pyspark.sql.types import StringType, DoubleType
from pyspark.storagelevel import StorageLevel

spark = SparkSession.builder \
    .appName('GarbageCollectionDemo') \
    .master('local[*]') \
    .config('spark.executor.memory', '2g') \
    .config('spark.memory.fraction', '0.6') \
    .config('spark.memory.storageFraction', '0.5') \
    .config('spark.executor.extraJavaOptions', '-XX:+UseG1GC') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Print key memory settings
for key in ['spark.executor.memory', 'spark.memory.fraction',
            'spark.memory.storageFraction', 'spark.executor.extraJavaOptions']:
    val = spark.conf.get(key, 'not set')
    print(f'  {key} = {val}')

print('\nSparkSession ready.')

In [ ]:
# Cell 2: GC pressure from object creation — UDFs vs built-in functions
# UDFs force data out of Tungsten's binary format into Python/JVM objects → GC pressure
# Built-in functions operate in Tungsten binary format (off-heap or on-heap binary) → minimal GC

import time
from pyspark.sql.functions import upper, length

df = spark.range(200_000).withColumn('name', (col('id') % 100).cast('string'))

# --- Approach 1: Python UDF (generates Java/Python objects for EVERY row = GC pressure) ---
udf_upper = udf(lambda s: s.upper() if s else None, StringType())

t0 = time.time()
df.withColumn('upper_name', udf_upper(col('name'))).count()
t1 = time.time()
print(f'Python UDF (GC-heavy):   {t1-t0:.2f}s')

# --- Approach 2: Built-in SQL function (Tungsten binary; no Python objects; near-zero GC) ---
t2 = time.time()
df.withColumn('upper_name', upper(col('name'))).count()
t3 = time.time()
print(f'Built-in function (GC-light): {t3-t2:.2f}s')

print()
print('Key insight: Python UDFs serialize each row to Python object and back → creates')
print('millions of short-lived JVM/Python objects → Young Gen fills fast → more Minor GC.')
print('Built-in functions use Tungsten code gen → no object creation → minimal GC.')

In [ ]:
# Cell 3: Storage levels and GC — serialised vs deserialized caching
# Caching with MEMORY_ONLY stores Java objects → large Old Gen footprint → GC pressure
# Caching with MEMORY_ONLY_SER stores byte arrays → compact; single object per partition → less GC

import sys

df_large = spark.range(500_000) \
    .withColumn('cat', (col('id') % 10).cast('string')) \
    .withColumn('val', (col('id') * 3.14).cast('double'))

# MEMORY_ONLY: stores deserialized Java objects — fast to access but GC-heavy
df_large.persist(StorageLevel.MEMORY_ONLY)
t0 = time.time()
c1 = df_large.count()
t_cache = time.time() - t0
t0 = time.time()
c2 = df_large.count()  # cache hit
t_hit = time.time() - t0
print(f'MEMORY_ONLY     — first action (cache): {t_cache:.2f}s  |  cache hit: {t_hit:.2f}s')
df_large.unpersist()

# MEMORY_ONLY_SER: stores serialized byte arrays — ~3x smaller; less GC but slower to deserialize
df_large.persist(StorageLevel.MEMORY_ONLY_SER)
t0 = time.time()
c3 = df_large.count()
t_cache2 = time.time() - t0
t0 = time.time()
c4 = df_large.count()  # cache hit
t_hit2 = time.time() - t0
print(f'MEMORY_ONLY_SER — first action (cache): {t_cache2:.2f}s  |  cache hit: {t_hit2:.2f}s')
df_large.unpersist()

print()
print('MEMORY_ONLY_SER is slower per access (deserialise overhead) but:')
print('  - Uses ~3x less heap space → fewer/shorter GC pauses')
print('  - Each partition is ONE byte array in Old Gen (easy to GC)')
print('  - Preferred when memory is tight or GC overhead is high')

In [ ]:
# Cell 4: Partition count and GC — too many partitions = too many objects
# Each partition = one task = one set of Java objects in the Executor heap
# Too many small partitions → excessive object creation overhead → GC thrash

df_base = spark.range(1_000_000)

# Too many small partitions (over-partitioned)
df_over = df_base.repartition(1000)   # 1000 partitions for 1M rows = 1000 rows/partition
t0 = time.time()
df_over.filter(col('id') % 2 == 0).count()
t1 = time.time()
print(f'Over-partitioned (1000 parts, ~1K rows each): {t1-t0:.2f}s')

# Well-sized partitions (~100MB/partition recommended)
df_good = df_base.repartition(8)     # 8 partitions = 125K rows/partition
t2 = time.time()
df_good.filter(col('id') % 2 == 0).count()
t3 = time.time()
print(f'Well-sized (8 parts, ~125K rows each):        {t3-t2:.2f}s')

print()
print('Rule of thumb for partition sizing:')
print('  Target: 100–200 MB per partition (uncompressed)')
print('  Target: 2–3x number of executor cores total')
print('  Too many small partitions → task scheduling + GC overhead dominates')
print('  Too few large partitions → executor OOM, skew, underutilised parallelism')

In [ ]:
# Cell 5: Off-heap memory configuration (bypasses JVM GC)
# When spark.memory.offHeap.enabled = true, Spark uses Tungsten's off-heap allocator
# Off-heap memory is managed by Spark directly — NOT subject to JVM GC

print('Off-heap configuration (requires restart with new SparkSession):')
print()
print('# In spark-submit or SparkSession.builder:')
print('.config("spark.memory.offHeap.enabled", "true")')
print('.config("spark.memory.offHeap.size", "2g")   # 2 GB off-heap per executor')
print()
print('Benefits of off-heap:')
print('  - JVM GC never sees off-heap memory → no GC pauses from Tungsten operations')
print('  - Reduces heap pressure → smaller Young/Old Gen → faster GC cycles')
print('  - Especially beneficial for large shuffle and join operations')
print()
print('Tradeoffs:')
print('  - Must be explicitly sized (does not auto-scale)')
print('  - Off-heap OOM crashes the JVM (no graceful fallback)')
print('  - Total memory = spark.executor.memory (heap) + spark.memory.offHeap.size + overhead')
print()

# Show current off-heap settings
offheap_enabled = spark.conf.get('spark.memory.offHeap.enabled', 'false')
offheap_size    = spark.conf.get('spark.memory.offHeap.size', '0')
print(f'Current: offHeap.enabled={offheap_enabled}, offHeap.size={offheap_size}')

In [ ]:
# Cell 6: Summary of GC best practices with demonstration

print('=== GC BEST PRACTICES SUMMARY ===')
print()

practices = [
    ('1. Use G1GC collector',
     'spark.executor.extraJavaOptions=-XX:+UseG1GC',
     'Predictable pause times; better for mixed young/old gen workloads'),
    ('2. Increase executor memory',
     'spark.executor.memory=4g (or higher)',
     'More heap space = less frequent GC; reduces Full GC frequency'),
    ('3. Use serialised storage levels',
     'persist(StorageLevel.MEMORY_ONLY_SER)',
     'Compact byte array per partition; less Old Gen pressure'),
    ('4. Avoid Python UDFs where possible',
     'Use built-in pyspark.sql.functions instead',
     'Eliminates row-level Java/Python object creation'),
    ('5. Right-size partitions',
     'Target 100-200 MB per partition',
     'Avoids excessive task overhead and per-task GC cycles'),
    ('6. Enable off-heap memory',
     'spark.memory.offHeap.enabled=true + spark.memory.offHeap.size=2g',
     'Shuffle/join data bypasses JVM GC entirely'),
    ('7. Reduce data in memory',
     'Select only needed columns; filter early; use Parquet',
     'Less data in memory = smaller heap footprint = less GC'),
    ('8. Unpersist when done',
     'df.unpersist() after last use',
     'Releases Old Gen blocks; prevents heap fragmentation'),
    ('9. Lower memory.fraction if GC is high',
     'spark.memory.fraction=0.5 (down from 0.6)',
     'Gives more heap to user/GC overhead reserve; reduces Old Gen pressure'),
    ('10. Enable speculative execution',
     'spark.speculation=true',
     'Tasks stalled by GC get a duplicate launched elsewhere; job completes faster'),
]

for name, config, benefit in practices:
    print(f'{name}')
    print(f'   Config:  {config}')
    print(f'   Benefit: {benefit}')
    print()

spark.stop()
print('SparkSession stopped.')

## 5. GC Tuning Configuration Reference

### Complete Spark GC Configuration Template

```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --num-executors 20 \
  --executor-cores 4 \
  --executor-memory 8g \
  --conf spark.executor.memoryOverhead=1g \
  --conf spark.memory.fraction=0.6 \
  --conf spark.memory.storageFraction=0.3 \
  --conf spark.memory.offHeap.enabled=true \
  --conf spark.memory.offHeap.size=4g \
  --conf 'spark.executor.extraJavaOptions=
    -XX:+UseG1GC
    -XX:G1HeapRegionSize=16m
    -XX:InitiatingHeapOccupancyPercent=35
    -XX:MaxGCPauseMillis=200
    -XX:+PrintGCDetails
    -XX:+PrintGCDateStamps
    -XX:+HeapDumpOnOutOfMemoryError' \
  my_app.py
```

### Memory Calculation Checklist

```
Total executor container memory requested from cluster manager:
= spark.executor.memory + spark.executor.memoryOverhead + spark.memory.offHeap.size

Example:
  spark.executor.memory      = 8g  (JVM heap)
  spark.executor.memoryOverhead = 1g  (off-heap: Python proc, network buffers)
  spark.memory.offHeap.size  = 4g  (Tungsten off-heap)
  Total per container        = 13g  ← this is what YARN/K8s allocates

Inside the 8g JVM heap:
  Reserved memory            = 300 MB  (fixed)
  Usable heap                = 8g - 300MB = ~7.7g
  Unified memory (execution + storage) = 7.7g * 0.6 = ~4.6g
  User memory                = 7.7g * 0.4 = ~3.1g
```

## 6. Real-World Scenarios

<details>
<summary>Scenario 1: Diagnosing GC Overhead via Spark UI Task Metrics</summary>

**Situation:**
A data engineering team notices that a nightly aggregation job takes 3 hours, up from 45 minutes last month.
The cluster size hasn't changed. Someone asks: "Could it be GC?"

**Investigation Steps:**
```
1. Open Spark History Server → select the failing job
2. Go to Stages tab → click the slow stage
3. Scroll to the Task Metrics table
4. Examine the "GC Time" column:
   - Task 0: Duration=45s, GC Time=40s (89%)  ← CRITICAL
   - Task 1: Duration=42s, GC Time=38s (90%)  ← CRITICAL
   - Task 2: Duration=41s, GC Time=37s (90%)  ← CRITICAL
   All tasks spending ~90% of time in GC!

5. Go to Executors tab → check Peak JVM Memory:
   Each executor peaked at 1.9 GB / 2 GB heap (95% full)

6. Check what changed last month:
   → Dataset grew from 10 GB to 80 GB
   → spark.executor.memory was never increased (still 2g)
   → executor heap is 95% full → constant Full GC
```

**Fix:**
```bash
# Increase executor memory to match dataset growth
spark-submit \
  --executor-memory 8g \
  --conf spark.executor.extraJavaOptions='-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35' \
  my_agg_job.py
```

**Expected Outcome:**
With 8g executor memory and G1GC, heap occupancy drops to ~40%.
GC time per task drops from 90% to <5%. Job completes in 55 minutes.

**Exam Sub-topic:** Identifying GC overhead; Spark UI task metrics; `spark.executor.memory`
</details>

<details>
<summary>Scenario 2: Python UDFs Causing GC Explosion — Migration to Built-in Functions</summary>

**Situation:**
A junior analyst wrote a PySpark job that applies 5 Python UDFs to a 500 GB DataFrame.
Each UDF does simple string operations (upper case, trim, concatenate).
The job is extremely slow with high GC overhead. The senior engineer suggests replacing UDFs.

**Before (GC-heavy):**
```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Each UDF serializes EVERY row to a Python object and back
clean_name = udf(lambda x: x.upper().strip() if x else None, StringType())
clean_city = udf(lambda x: x.strip().title() if x else None, StringType())
full_name  = udf(lambda f, l: f'{f} {l}' if f and l else None, StringType())

df = df.withColumn('clean_name', clean_name(col('name'))) \
       .withColumn('clean_city', clean_city(col('city'))) \
       .withColumn('full_name', full_name(col('first'), col('last')))
```

**After (GC-light — built-in functions operate in Tungsten binary format):**
```python
from pyspark.sql.functions import upper, trim, initcap, concat_ws

df = df.withColumn('clean_name', upper(trim(col('name')))) \
       .withColumn('clean_city', initcap(trim(col('city')))) \
       .withColumn('full_name', concat_ws(' ', col('first'), col('last')))
```

**Expected Outcome:**
GC time per task drops from 60% to <2%. Job runtime drops from 4 hours to 25 minutes.
No Python serialisation overhead. Tungsten code-gen handles all operations in binary.

**Exam Sub-topic:** Python UDF GC impact; built-in functions; Tungsten execution engine
</details>

<details>
<summary>Scenario 3: Serialised Caching to Reduce Old Gen GC Pressure</summary>

**Situation:**
A recommendation system caches a 15 GB item feature DataFrame in executor memory
and uses it across 50 ML model scoring iterations. With default `cache()` (MEMORY_AND_DISK),
frequent Full GC pauses are observed after iteration 20 because the Old Gen fills up
with large deserialized objects from the cached DataFrame.

**Problem Code:**
```python
item_features = spark.read.parquet('/data/item_features/')  # 15 GB
item_features.cache()  # MEMORY_AND_DISK = deserialized objects in Old Gen
item_features.count()  # materialise

for iteration in range(50):
    # Each iteration reads item_features — but GC pauses grow worse each time
    scores = model.score(candidates.join(item_features, on='item_id'))
    scores.write.parquet(f'/output/scores_{iteration}/', mode='overwrite')
```

**Fix — Serialised caching:**
```python
from pyspark.storagelevel import StorageLevel

item_features = spark.read.parquet('/data/item_features/')
# MEMORY_AND_DISK_SER: each partition stored as a compact byte array (one object in Old Gen)
# ~3x smaller footprint → Old Gen occupancy drops from 90% to 30% → no Full GC
item_features.persist(StorageLevel.MEMORY_AND_DISK_SER)
item_features.count()  # materialise

for iteration in range(50):
    scores = model.score(candidates.join(item_features, on='item_id'))
    scores.write.parquet(f'/output/scores_{iteration}/', mode='overwrite')

item_features.unpersist()  # release after all iterations
```

**Expected Outcome:**
Old Gen footprint drops by ~3x. Full GC pauses disappear. All 50 iterations complete without
GC overhead increasing. Total job time reduced by 40%.

**Exam Sub-topic:** Storage level selection; serialised caching; Old Gen GC; `persist()` vs `cache()`
</details>

<details>
<summary>Scenario 4: Off-Heap Memory for Shuffle-Heavy Join to Eliminate GC</summary>

**Situation:**
A fraud detection pipeline runs a SortMergeJoin between a 200 GB transactions table and
a 50 GB merchant table. The shuffle and join hash tables consume most of the executor heap,
causing repeated Full GC pauses. Enabling off-heap memory moves Tungsten's shuffle buffers
and join tables out of the JVM heap entirely.

**Configuration:**
```bash
spark-submit \
  --executor-memory 8g \
  --conf spark.memory.offHeap.enabled=true \
  --conf spark.memory.offHeap.size=6g \
  --conf spark.executor.memoryOverhead=2g \
  --conf 'spark.executor.extraJavaOptions=-XX:+UseG1GC -XX:G1HeapRegionSize=16m' \
  fraud_detection.py
```

```python
# In-code equivalent
spark = SparkSession.builder \
    .config('spark.executor.memory', '8g') \
    .config('spark.memory.offHeap.enabled', 'true') \
    .config('spark.memory.offHeap.size', str(6 * 1024 * 1024 * 1024)) \
    .config('spark.memory.fraction', '0.5') \
    .getOrCreate()

# With off-heap enabled:
# - Shuffle sort buffers → off-heap (no GC)
# - Join hash tables    → off-heap (no GC)
# - JVM heap now used only for task overhead + user data (~40% of former pressure)
transactions = spark.read.parquet('/data/transactions/')
merchants    = spark.read.parquet('/data/merchants/')
result = transactions.join(merchants, on='merchant_id')
result.write.parquet('/output/fraud_features/', mode='overwrite')
```

**Expected Outcome:**
JVM GC overhead drops from 35% per task to <3%. Full GC events: 0.
Total memory per executor: 8g (heap) + 2g (overhead) + 6g (off-heap) = 16g container.
Job runtime reduced by 50%.

**Exam Sub-topic:** Off-heap memory; `spark.memory.offHeap`; Tungsten; shuffle memory
</details>

<details>
<summary>Scenario 5: Reducing GC by Lowering spark.memory.fraction</summary>

**Situation:**
A Spark job processes event logs with complex Python UDFs that create many Python dictionaries
and lists per row (user memory pressure). At the same time, the job caches intermediate
DataFrames (storage memory pressure). Old Gen fills with both cached data and user objects,
causing constant Full GC.

**Analysis:**
```
Default allocation (executor heap = 10g, reserved = 0.3g, usable = 9.7g):
  Unified memory  = 9.7g * 0.6 = 5.8g  (execution + storage)
  User memory     = 9.7g * 0.4 = 3.9g  (Python objects)
  → User memory is full → spills into unified → execution memory squeezed → Full GC
```

**Fix — Lower memory.fraction to give more space to user memory:**
```python
spark = SparkSession.builder \
    .config('spark.executor.memory', '10g') \
    .config('spark.memory.fraction', '0.4') \
    .config('spark.memory.storageFraction', '0.3') \
    .config('spark.executor.extraJavaOptions',
            '-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=30') \
    .getOrCreate()

# New allocation:
#   Unified memory  = 9.7g * 0.4 = 3.9g  (smaller; less Old Gen pressure from cached data)
#   User memory     = 9.7g * 0.6 = 5.8g  (larger; plenty of space for Python objects)
#   Result: Old Gen occupancy drops; GC pauses disappear

# Also: replace Python UDFs with pandas UDFs (vectorized) to further reduce object creation
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(returnType='double')
def score_events(event_series: pd.Series) -> pd.Series:
    # Operates on entire Pandas Series (vectorised) — far fewer JVM objects than row-level UDF
    return event_series.apply(lambda e: len(e) * 0.1)
```

**Expected Outcome:**
User memory pressure relieved. GC time drops from 70% to 8% per task.
Pandas UDF replaces row-level Python UDF — 10x fewer object allocations.
Job completes in 30 minutes vs 3 hours.

**Exam Sub-topic:** `spark.memory.fraction`; user vs unified memory; pandas UDFs vs Python UDFs; GC tuning
</details>

## 7. Exam Cheat Sheet

### Key Facts

| Fact | Value / Detail |
|------|----------------|
| Spark Executors run in | **JVM processes** — JVM GC applies |
| Recommended GC collector | **G1GC** (`-XX:+UseG1GC`) |
| Set GC flags via | `spark.executor.extraJavaOptions` |
| `spark.memory.fraction` default | **0.6** — fraction of heap for execution + storage |
| `spark.memory.storageFraction` default | **0.5** — within unified memory, protected for storage |
| Reserved memory | Fixed **300 MB** per executor (internal Spark objects) |
| Spot GC problem in Spark UI | Stages → Task Metrics → **GC Time column** |
| GC time > 10% of task duration | **GC pressure** — investigate |
| GC time > 25% of task duration | **Serious problem** — tune memory |
| `cache()` vs `MEMORY_ONLY_SER` for GC | `MEMORY_ONLY_SER` is **~3x smaller** → less GC |
| Off-heap bypasses GC? | **Yes** — `spark.memory.offHeap.enabled=true` |
| Python UDFs and GC | UDFs create **per-row Java/Python objects** → high GC; use built-in functions instead |
| Too many small partitions | Excessive per-task GC cycles; target **100–200 MB/partition** |
| `unpersist()` GC benefit | Releases Old Gen blocks; prevents fragmentation |

---

## 8. Practice Q&A

<details>
<summary>Q1: Why does GC matter in Spark, and where does it happen?</summary>

**A:** Spark Executors are **JVM processes**. All data processing objects (rows, intermediate results, cached RDD blocks, broadcast variables) live on the JVM heap and are subject to JVM Garbage Collection.

GC matters because:
1. During a **GC pause**, ALL threads in the Executor JVM stop (stop-the-world) — tasks are frozen
2. **Minor GC** (Young Gen) is frequent and fast — usually <100ms
3. **Full GC** (Old Gen) is infrequent but slow — can pause for seconds or minutes
4. Excessive GC → tasks appear stalled → Spark may mark executor as lost → job fails
5. `OutOfMemoryError: GC overhead limit exceeded` — JVM kills itself when GC takes >98% of time
</details>

<details>
<summary>Q2: How do you identify a GC problem in a running or completed Spark job?</summary>

**A:**
1. **Spark UI → Stages tab → Task Metrics**: look at the **GC Time** column. If GC Time > 10–25% of Task Duration, there is a GC issue.
2. **Spark UI → Executors tab**: check **Peak JVM Memory**. If close to `spark.executor.memory`, the heap is too small.
3. **Executor logs** (with `-XX:+PrintGCDetails`): look for `[Full GC ...]` entries with pause times >1 second.
4. **Ganglia/Prometheus**: monitor JVM heap utilisation metrics in real time.
</details>

<details>
<summary>Q3: What is spark.memory.fraction and how does lowering it help with GC?</summary>

**A:** `spark.memory.fraction` (default 0.6) controls what fraction of the usable JVM heap is reserved for Spark's **unified memory** (execution + storage). The remaining 40% is **user memory** — for Python objects, UDF closures, and non-Spark JVM objects.

If user objects (especially from complex Python UDFs) overflow user memory into the unified region, execution and storage get squeezed, causing OOM or increased GC pressure.

**Lowering** `spark.memory.fraction` to 0.4–0.5:
- Gives more heap to user memory → Python/UDF objects have more room
- Reduces unified memory → smaller Old Gen footprint from cached data → less Full GC
- Trade-off: less execution memory may cause more shuffle spills to disk
</details>

<details>
<summary>Q4: How does off-heap memory (spark.memory.offHeap) help with GC?</summary>

**A:** When `spark.memory.offHeap.enabled=true`, Spark's Tungsten engine allocates memory **outside the JVM heap** using direct (native) memory. This memory is managed by Spark itself, not the JVM GC.

Benefits:
- Shuffle sort buffers, join hash tables, and aggregate hash maps are allocated off-heap
- The JVM GC never sees this memory → **no GC pauses** from Tungsten operations
- JVM heap can be kept smaller → faster, more predictable GC

Configuration: `spark.memory.offHeap.size` must be set (e.g., `4g`). Total executor container memory = heap + overhead + offheap.
</details>

<details>
<summary>Q5: What are the top 3 code-level changes that reduce GC pressure in PySpark?</summary>

**A:**
1. **Replace Python UDFs with built-in functions** — Python UDFs serialise every row to a Python object and back; built-in functions operate on Tungsten binary data with zero object creation per row. Typical impact: 80–90% reduction in GC overhead.

2. **Use serialised storage levels for caching** — `persist(StorageLevel.MEMORY_ONLY_SER)` or `MEMORY_AND_DISK_SER` stores each partition as a single byte array (one Old Gen object) instead of thousands of deserialized row objects. Typical heap saving: ~3x.

3. **Select/project early and minimise data size** — use `select()` to keep only needed columns, `filter()` as early as possible, and write/read Parquet (columnar, compressed). Smaller data → smaller heap → less GC.

Bonus: Use **pandas UDFs** (`@pandas_udf`) instead of row-level Python UDFs when a UDF is unavoidable — pandas UDFs are vectorized (operate on Series), creating far fewer JVM objects.
</details>